In [2]:
#bash
#!pip install -q flwr[simulation] flwr_datasets[vision] torch torchvision matplotlib tensorflow scikit-learn 

#zsh
#!pip install -q flwr\[simulation\] flwr_datasets\[vision\] torch torchvision matplotlib tensorflow scikit-learn 

In [3]:
# Import necessary libraries
import flwr as fl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import logging
import random
import ray

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers
from datasets.utils.logging import disable_progress_bar
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set up logging
logging.basicConfig(level=logging.INFO)

DEVICE = torch.device("cpu")  # Try "cuda" to train on GPU
#DEVICE = torch.device("cuda")  # Try "cuda" to train on GPU
print(
    f"Training on {DEVICE} using PyTorch {torch.__version__} and Flower {fl.__version__}"
)
disable_progress_bar()

2024-07-17 20:00:50.153183: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-17 20:00:50.176625: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-17 20:00:50.176678: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-17 20:00:50.191020: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-17 20:00:51.023691: W tensorflow/compiler/tf

Training on cpu using PyTorch 2.3.1+cu121 and Flower 1.9.0


In [4]:
# Define the path to the dataset and the sequence length
#PATH = '/Users/tuliopolido/DBs/vivabem_aerobic_65users.csv'
#PATH = 'D:/DBs/vivabem_aerobic_65users.csv'
PATH = '/home/tulionpl/DBs/vivabem_aerobic_65users_2.csv'
SEQUENCE_LENGTH = 7
NUM_CLASSES = 3
BATCH_SIZE = 16
NUM_EPOCHS = 5    # You can change this to the desired number of epochs
LEARNING_RATE = 0.005    # You can change this to the desired number of epochs

# Define the number of clients and epochs
NUM_CLIENTS = 5  # You can change this to the desired number of clients

# Define client's resources
CLIENT_RESOURCES = {'num_cpus': 2.0}

ID_COLUMN = 'int_id'

In [5]:
# Function to load and preprocess the dataset
def load_dataset(path, sequence_length=5, num_clients=10):
    """
    Load and preprocess the dataset.

    Args:
        path (str): The path to the dataset CSV file.
        sequence_length (int): The length of the sequences to create.

    Returns:
        dict: A dictionary where keys are user IDs and values are tuples (X_client, y_client)
              representing the features and labels for each client.
        int: The number of features in the dataset.
    """
    print("Loading dataset...")
    # Load the dataset
    data = pd.read_csv(path)
    data[ID_COLUMN] = data[ID_COLUMN].astype(str)
    print(data.columns)

    print("Pre-processing...")
    # Preprocess the data
    X = data[['x-axis', 'y-axis', 'z-axis', 'GYR_X', 'GYR_Y', 'GYR_Z']]
    y = data['activity']

    print("Normalizing...")
    # Normalize the features
    X = (X - X.mean()) / X.std()

    # Convert data into sequences
    def create_sequences(X, y, sequence_length):
        """
        Convert data into sequences.

        Args:
            X (pd.DataFrame): The input features.
            y (pd.Series): The target labels.
            sequence_length (int): The length of the sequences to create.

        Returns:
            np.ndarray: The input sequences.
            np.ndarray: The target labels for each sequence.
        """
        print(X.shape)
        print(X[0:5])
        print(y.shape)
        print(y[0:5])

        Xs, ys = [], []
        for i in range(len(X) - sequence_length):
            Xs.append(X.iloc[i:(i + sequence_length)].values)
            ys.append(y.iloc[i + sequence_length])

        Xs = np.array(Xs)
        ys = np.array(ys)
        print(Xs.shape)
        print(Xs[0])
        print(ys.shape)
        print(ys[0])
        return Xs, ys

    print("Splitting data...")
    # Split the data into clients based on user-id
    clients = list(data[ID_COLUMN].unique())
    random.shuffle(clients)  # Shuffle the clients
    selected_clients = clients[:num_clients]  # Limit the number of clients
    print('Selected clients:', selected_clients)

    data_splits = {}
    for client_id in selected_clients:
        client_data = data[data[ID_COLUMN] == client_id]
        X_client, y_client = create_sequences(client_data[['x-axis', 'y-axis', 'z-axis', 'GYR_X', 'GYR_Y', 'GYR_Z']], client_data['activity'], sequence_length)
        data_splits[client_id] = (X_client, y_client)
    
    return data_splits, X.shape[1]


In [6]:
# Load dataset
data_splits, input_dim = load_dataset(PATH, SEQUENCE_LENGTH, NUM_CLIENTS)
input_shape = (SEQUENCE_LENGTH, input_dim)
print(input_shape)
print(data_splits.keys())

Loading dataset...
Index(['int_id', 'user-id', 'timestamp', 'x-axis', 'y-axis', 'z-axis', 'GYR_X',
       'GYR_Y', 'GYR_Z', 'activity'],
      dtype='object')
Pre-processing...
Normalizing...
Splitting data...
Selected clients: ['1', '0']
(69649, 6)
       x-axis    y-axis    z-axis      GYR_X      GYR_Y      GYR_Z
980 -9.052477 -3.100491  1.450886    1.19000  -0.490000  -1.890000
981 -9.006987 -3.450045  0.433351 -169.82004 -34.090008 -20.090006
982 -9.255983 -3.347094  0.318429 -160.16003 -33.460007 -17.150003
983 -9.576807 -3.110068  0.181959 -148.40002 -32.970005 -13.090003
984 -9.653421 -2.932897  0.124498 -136.08003 -32.480007  -8.400002
(69649,)
980    0
981    0
982    0
983    0
984    0
Name: activity, dtype: int64
(69642, 7, 6)
[[-9.05247700e+00 -3.10049130e+00  1.45088620e+00  1.19000030e+00
  -4.90000130e-01 -1.89000030e+00]
 [-9.00698700e+00 -3.45004460e+00  4.33350500e-01 -1.69820040e+02
  -3.40900080e+01 -2.00900060e+01]
 [-9.25598300e+00 -3.34709400e+00  3.18428800e-01

In [7]:
class CNNNet(nn.Module):
    def __init__(self, input_shape, num_classes):
        super(CNNNet, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_shape[1], out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * (input_shape[0] // 4), 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [8]:
def train(model, client_data, num_epochs=10, batch_size=32, learning_rate=0.001):
    """
    Train the CNN model on data from a single client using CrossEntropyLoss.

    Args:
        model (nn.Module): The CNN model.
        client_data (tuple): A tuple (X_client, y_client) representing the features and labels for the client.
        num_epochs (int): The number of epochs to train.
        batch_size (int): The batch size for training.
        learning_rate (float): The learning rate for the optimizer.

    Returns:
        None
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    X_client, y_client = client_data
    dataset = TensorDataset(torch.tensor(X_client, dtype=torch.float32).permute(0, 2, 1), torch.tensor(y_client, dtype=torch.long))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        
        for X_batch, y_batch in dataloader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(dataloader):.4f}")


In [9]:
def test(model, client_data):
    """
    Test the CNN model on data from a single client using various metrics.

    Args:
        model (nn.Module): The CNN model.
        client_data (tuple): A tuple (X_client, y_client) representing the features and labels for the client.

    Returns:
        dict: A dictionary containing the Cross Entropy Loss, accuracy, precision, F1 score, and confusion matrix.
    """
    model.eval()
    
    X_client, y_client = client_data
    X_tensor = torch.tensor(X_client, dtype=torch.float32).permute(0, 2, 1)  # Permutar para (num_samples, features, timesteps)
    y_tensor = torch.tensor(y_client, dtype=torch.long)
    
    criterion = nn.CrossEntropyLoss()

    total_loss = 0.0
    y_true = []
    y_pred = []

    with torch.no_grad():
        outputs = model(X_tensor)
        loss = criterion(outputs, y_tensor)
        total_loss += loss.item()
        
        y_true = y_tensor.numpy()
        y_pred = outputs.argmax(dim=1).numpy()  # Obter a classe predita tomando o argmax

    cross_entropy_loss = total_loss / len(X_client)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    conf_matrix = confusion_matrix(y_true, y_pred)

    return cross_entropy_loss, accuracy, precision, f1, conf_matrix


In [12]:
model = CNNNet(input_shape, NUM_CLASSES)

# Select a client ID to train and test on
client_id = list(data_splits.keys())[0]  # Replace with desired client ID
print(client_id)
client_data = data_splits[client_id]
print(len(client_data[0]), len(client_data[1]))

1
69642 69642


In [13]:
# Train the model on the selected client data
train(model, client_data, num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE)

# Test the model on the selected client data
cross_entropy_loss, accuracy, precision, f1, conf_matrix = test(model, client_data)

Epoch [1/5], Loss: 0.5678
Epoch [2/5], Loss: 0.3624
Epoch [3/5], Loss: 0.4398
Epoch [4/5], Loss: 0.2539
Epoch [5/5], Loss: 0.3116


In [14]:
print(f"cross_entropy_loss: {cross_entropy_loss}")
print(f"accuracy: {accuracy}")
print(f"precision: {precision}")
print(f"f1: {f1}")
print(f"conf_matrix: \n {conf_matrix}")

cross_entropy_loss: 3.86183484945842e-06
accuracy: 0.9217569857270038
precision: 0.928496106313435
f1: 0.9206868449650368
conf_matrix: 
 [[18888   405    20]
 [ 1809 24493  3193]
 [    0    22 20812]]
